<a href="https://colab.research.google.com/github/shris2810/Langchain-Langgraph/blob/main/Simple_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
import os

# Hugging Face access token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [1]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [2]:
pip install -U youtube-transcript-api langchain-text-splitters langchain-openai langchain-community langchain-core faiss-cpu

In [17]:
pip install -U langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

# LangChain Modular Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_3121/2904343161.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [ ]:
!pip install

Fetching vedio transcript

In [12]:
video_id = "fNk_zzaMoSs"

try:
    # 1. Instantiate the client
    ytt = YouTubeTranscriptApi()

    # 2. Fetch the transcript (pass languages as a list/tuple)
    transcript_list = ytt.fetch(video_id, languages=["en"])

    # 3. Convert to the list of dictionaries [{'text': ..., 'start': ..., ...}]
    raw_data = transcript.to_raw_data()

    # 4. Flatten to plain text for LangChain
    transcript = " ".join(item["text"] for item in raw_data)
    print("Transcript fetched successfully!")

except (TranscriptsDisabled) as e:
    print(f"Could not retrieve transcript: {e}")

Transcript fetched successfully!


In [15]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Translated by Grant Sanderson. Submit corrections at criblate.com]', start=0.0, duration=10.92), FetchedTranscriptSnippet(text='The fundamental, root-of-it-all building block for linear algebra is the vector.', start=10.92, duration=4.3), FetchedTranscriptSnippet(text="So it's worth making sure that we're all on the same page about what exactly a vector is.", start=15.72, duration=4.12), FetchedTranscriptSnippet(text='You see, broadly speaking, there are three distinct but related ideas about vectors,', start=20.38, duration=3.851), FetchedTranscriptSnippet(text="which I'll call the physics student perspective,", start=24.231, duration=2.247), FetchedTranscriptSnippet(text="the computer science student perspective, and the mathematician's perspective.", start=26.478, duration=3.622), FetchedTranscriptSnippet(text='The physics student perspective is that vectors are arrows pointing in space.', start=30.88, duration=3.52), Fetch

Indexing

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [19]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Downloads once (~90MB) and runs locally on CPU/GPU
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Build your FAISS index
vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
vector_store.index_to_docstore_id

{0: '686708b2-50cd-47a7-8359-48a1ea5ca1dd',
 1: 'ef2d3278-6846-48e6-b13d-5f6229100716',
 2: '2c4b7dda-6acf-447f-bfc7-551cc796fb9e',
 3: '8e5fedf1-59ce-494f-aff0-19b7578561f3',
 4: '6c961a6c-3b7e-4f9d-b016-7c9db10da2bb',
 5: '8aab53e6-c05b-4a5d-aead-680a40d50458',
 6: '54bb4078-c385-4750-8f29-ce7ddc531d2a',
 7: '9480c6ad-84d8-4c98-af33-41413514cd2b',
 8: 'cb69e6a6-49d0-4602-8c9f-5d0b8c077500',
 9: '15caf2e3-dc2e-42e0-b6d7-ec74c8634a3c',
 10: '250e2238-2ae7-4820-b07a-3436cf1dbb4c',
 11: '55b7af5c-da24-4af2-b2b8-54927829821e',
 12: '7a4e1404-e5db-4560-a4dc-0e473069a65f'}

In [21]:
vector_store.get_by_ids(['7a4e1404-e5db-4560-a4dc-0e473069a65f'])

[Document(id='7a4e1404-e5db-4560-a4dc-0e473069a65f', metadata={}, page_content="and the manipulation of space using numbers that can be crunched and run through a computer. When I do math-y animations, for example, I start by thinking about what's actually going on in space, and then get the computer to represent things numerically, thereby figuring out where to place the pixels on the screen. And doing that usually relies on a lot of linear algebra understanding. So there are your vector basics, and in the next video I'll start getting into some pretty neat concepts surrounding vectors like span, bases, and linear dependence. See you then!")]

Retriver

In [22]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

In [23]:
retriever.invoke("what is vectors?")

[Document(id='686708b2-50cd-47a7-8359-48a1ea5ca1dd', metadata={}, page_content="[Translated by Grant Sanderson. Submit corrections at criblate.com] The fundamental, root-of-it-all building block for linear algebra is the vector. So it's worth making sure that we're all on the same page about what exactly a vector is. You see, broadly speaking, there are three distinct but related ideas about vectors, which I'll call the physics student perspective, the computer science student perspective, and the mathematician's perspective. The physics student perspective is that vectors are arrows pointing in space. What defines a given vector is its length and the direction it's pointing, but as long as those two facts are the same, you can move it all around, and it's still the same vector. Vectors that live in the flat plane are two-dimensional, and those sitting in broader space that you and I live in are three-dimensional. The computer science perspective is that vectors are ordered lists of nu

Prompt building

In [24]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [26]:
query = "what is vectors?"
retrive_doc = retriever.invoke(query)

In [30]:
retrive_doc
# we need to sent text in context not doc object

[Document(id='686708b2-50cd-47a7-8359-48a1ea5ca1dd', metadata={}, page_content="[Translated by Grant Sanderson. Submit corrections at criblate.com] The fundamental, root-of-it-all building block for linear algebra is the vector. So it's worth making sure that we're all on the same page about what exactly a vector is. You see, broadly speaking, there are three distinct but related ideas about vectors, which I'll call the physics student perspective, the computer science student perspective, and the mathematician's perspective. The physics student perspective is that vectors are arrows pointing in space. What defines a given vector is its length and the direction it's pointing, but as long as those two facts are the same, you can move it all around, and it's still the same vector. Vectors that live in the flat plane are two-dimensional, and those sitting in broader space that you and I live in are three-dimensional. The computer science perspective is that vectors are ordered lists of nu

In [27]:
context_text = "\n\n".join(doc.page_content for doc in retrive_doc)

In [28]:
final_prompt = prompt.invoke({'context':context_text,'question':query})

Generation

In [35]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
) # type: ignore

model = ChatHuggingFace(
    llm=llm
)

In [36]:
answer = model.invoke(final_prompt)


In [38]:
answer.content

"According to the transcript, there are three distinct but related ideas about vectors:\n\n1. **Physics student perspective**: Vectors are arrows pointing in space, defined by their length and direction.\n2. **Computer science perspective**: Vectors are ordered lists of numbers, where the order matters.\n3. **Mathematician's perspective**: A vector can be anything where there's a sensible notion of adding two vectors and multiplying a vector by a number.\n\nIn a specific context, a vector can also be thought of as an arrow inside a coordinate system, like the xy-plane, with its tail sitting at the origin."

Chain Building

In [41]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [42]:
def format_docs(retrived_doc):
  context_text = "\n\n".join(doc.page_content for doc in retrived_doc)
  return context_text


In [43]:
parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})



In [44]:
parallel_chain.invoke("what is vector")

{'context': "[Translated by Grant Sanderson. Submit corrections at criblate.com] The fundamental, root-of-it-all building block for linear algebra is the vector. So it's worth making sure that we're all on the same page about what exactly a vector is. You see, broadly speaking, there are three distinct but related ideas about vectors, which I'll call the physics student perspective, the computer science student perspective, and the mathematician's perspective. The physics student perspective is that vectors are arrows pointing in space. What defines a given vector is its length and the direction it's pointing, but as long as those two facts are the same, you can move it all around, and it's still the same vector. Vectors that live in the flat plane are two-dimensional, and those sitting in broader space that you and I live in are three-dimensional. The computer science perspective is that vectors are ordered lists of numbers. For example, let's say you were doing some analytics about h

In [45]:
parser = StrOutputParser()

In [46]:
main_chain = parallel_chain | prompt | model | parser

In [47]:
main_chain.invoke("can you summarize the vedio")

"I don't know. The transcript doesn't mention a video, but based on the provided text, here's a summary:\n\nThe text discusses vectors, their geometric representation as arrows, and their algebraic representation as ordered pairs of numbers. It explains how to perform vector addition and multiplication by a number, and introduces the concept of coordinate systems, including the xy-plane and the addition of a third axis, the z-axis, to represent three-dimensional space. The text also mentions that vectors will be a key part of linear algebra."